# 16 pamoka – Keičiamų agentų diegimas su Microsoft Foundry

Šiame užrašų knygelėje jūs kuriate **gamybinį klientų aptarnavimo agentą** išgalvotai įmonei **Contoso**. Skirtingai nuo ankstesnių pamokų, čia svarbiausias nėra agento mąstymo ciklas – tai viskas, kas yra *aplink* jį, kas leidžia agentui saugiai veikti plačiu mastu:

1. **Įrankių iškvietimas** – užsakymų paieška ir bilietų kūrimas.
2. **RAG** – politikos atsakymai iš žinių bazės.
3. **Atmintis** – prisiminti klientą per pokalbio raundus.
4. **Modelio maršrutizavimas** – paprastus užklausimus siųsti mažam modeliui, sudėtingus – dideliam modeliui.
5. **Atsakymų talpinimas** – aptarnauti pasikartojančius klausimus be modelio kvietimo.
6. **Žmogaus patvirtinimas** – grąžinimai virš ribos sustabdomi, kol gaunamas patvirtinimas.
7. **Vertinimo barjeras** – offline testų rinkinys, blokuojantis prastą išleidimą.
8. **Stebėjimas** – OpenTelemetry sekimas kiekvienos užklausos metu.

Kiekviena dalis yra savarankiška ir paleidžiama. Skaitykite kiekvieną eilutę – gamybiniai elementai sąmoningai laikomi mažais.


## Diegimas

Prieš paleisdami šį užrašų knygelę, įsitikinkite, kad turite:

1. **Microsoft Foundry projektą** su išdiegta pokalbių modeliu (pvz., `gpt-5-mini`).
2. **Prisijungę per Azure CLI** — terminale vykdykite `az login`.
3. **Nustatytus reikiamus aplinkos kintamuosius:**
   - `AZURE_AI_PROJECT_ENDPOINT` — jūsų Microsoft Foundry projekto pabaigos taškas.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jūsų išdiegto modelio pavadinimas.

RAG skyrius naudoja **Azure AI Search**, kai nustatyti `AZURE_SEARCH_SERVICE_ENDPOINT` ir `AZURE_SEARCH_API_KEY`, o jei šie nėra nustatyti, naudoja atmintinę paiešką, kad užrašų knygelė veiktų be Search išteklių.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Įrankiai

Gamybos įrankiai atlieka tikrą darbą su tikromis sistemomis. Čia mes modeliuojame užsakymų duomenų bazę ir bilietų sistemą naudodami paprastas Python funkcijas. Dekoratorius `@tool` atveria jas agentui.

Atkreipkite dėmesį, kad `issue_refund` naudoja `approval_mode="always_require"` grąžinimams, viršijantiems ribą – tai žmogaus įsikišimo primityvas, kurį diegiame vėliau.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Politikos žinių bazė

Politikos klausimai („koks jūsų prekių grąžinimo laikotarpis?“) turėtų būti atsakomi iš autoritetingo šaltinio, o ne iš modelio atminties. Mes apgaubiam mažą žinių bazę kaip paieškos įrankį.

Produkcijoje tai yra **Azure AI Search**; čia mes suteikiame raktinių žodžių paiešką atmintyje, kad užrašų knygelė veiktų bet kur, ir automatiškai pereiname prie Azure AI Search, kai yra aplinkos kintamieji.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Atmintis

Pagalbos agentas, kuris pamiršta, su kuo kalba, yra prastas pagalbos agentas. Mes laikome nedidelį kiekvieno kliento profilio saugyklą ir įtraukiame trumpą santrauką į agente pateikiamas instrukcijas. Gamyboje tai yra atminties paslauga (žr. 13 pamoką); čia žodynas daro šį modelį matomą.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 ir 5. Modelių maršrutizavimas ir atsakymų talpinimas

Du kaštų svertai sujungti į vieną užklausos apdorojimo modulį:

- **Maršrutizavimas**: pigus heuristinis klasifikatorius nusprendžia, ar užklausa reikalauja mažo, ar didelio modelio.
- **Talpinimas**: normalizuoti pasikartojantys klausimai aptarnaujami tiesiai iš talpyklos, nekviečiant modelio.

Čia klasifikatorius sąmoningai paprastas. Gamyboje jūs jį patikrintumėte pagal srautą ir galėtumėte pakeisti Foundry Model Router.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-5-nano
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-5-mini

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 ir 8. Agentas, Žmogiškas Patvirtinimas ir Stebimumas

Dabar sudedame agentą iš aukščiau esančių įrankių ir apgaubiam kiekvieną užklausą OpenTelemetry span'u. Funkcija `handle_support_request` yra gamybos užklausų tvarkyklė: cache → route → trace → run → cache.

Žmogiškas patvirtinimas yra tvarkomas sistemos: kadangi `issue_refund` turi `approval_mode="always_require"`, vykdymas sustoja ir pateikia patvirtinimo užklausą, kurią mes aiškiai išsprendžiame.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

# Build one agent per model tier so we can route by cost. The current agent-framework
# selects the model on the client, so each tier gets its own FoundryChatClient.
_TOOLS = [get_order_status, open_ticket, search_policies, issue_refund]
_agents_by_model: dict[str, object] = {}


def agent_for(model_name: str):
    if model_name not in _agents_by_model:
        client = FoundryChatClient(
            project_endpoint=endpoint,
            model=model_name,
            credential=AzureCliCredential(),
        )
        _agents_by_model[model_name] = client.as_agent(
            name="ContosoSupportAgent",
            instructions=SUPPORT_INSTRUCTIONS,
            tools=_TOOLS,
        )
    return _agents_by_model[model_name]


# Default agent (used by the evaluation gate, which does not route).
support_agent = agent_for(SMALL_MODEL)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await agent_for(chosen_model).run(prompt)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Vertinimo vartai

Tai yra išleidimo vartai iš pamokos: nešiojamas testų rinkinys įvertina agentą, o diegimas vykdomas tik tuo atveju, jei praėjimo rodiklis viršija tam tikrą ribą. Čia vertintojas yra paprastas raktinių žodžių sutapimų tikrinimas, kad užrašų knygelė būtų savarankiška; gamyboje naudotumėte LLM kaip teisėją arba įvertinimo sistemą (žr. Pamoką 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Visą sujungti: simuliuotas paleidimas

Žemiau esanti ląstelė rodo visą ciklą, kurį pamoka aprašo: vykdyti vertinimo vartus ir „diegi“ tik tada, jei jis praeina. Tai yra modelis, kurį paleistumėte CI prieš reklamuodami agento versiją Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Santrauka

Jūs sukūrėte gamybai parengtą klientų aptarnavimo agentą, kuriame įtraukti visi veiklos aspektai:

- **Įrankiai, RAG ir atmintis** suteikia agentui galimybes ir kontekstą.
- **Modelių maršrutizavimas ir kešavimas** kontroliuoja delsą bei kainą.
- **Žmogiškas patvirtinimas** saugo nuo didelės rizikos veiksmų, kaip dideli grąžinimai.
- **Įvertinimo filtras** užkirsti kelią blogiems leidimams prieš juos išleidžiant.
- **Sekimas** padaro kiekvieną užklausą stebimą.

### Iššūkis

Išplėskite šį agentą, kad:

1. **Palaikytų kelis modelius** — pridėkite trečią "mąstymo" lygį ir nukreipkite eskalacijas/skundus jam.
2. **Pridėtumėte įvertinimo filtrus** — išplėskite `TEST_CASES` įtraukdami scenarijus dėl grąžinimo patvirtinimo ir patvirtinkite, kad filtras sugaudo regresijas.
3. **Pridėtumėte maršrutizavimą, atsižvelgiant į kainą** — sekite numatomą užklausos kainą (maža, didelė ar kešuota) ir po mišrios užklausų partijos atspausdinkite kainų ataskaitą.

Kitoje pamokoje jūs pasuksite priešinga kryptimi ir įkursite agentą visiškai savo kompiuteryje su Microsoft Foundry Local ir Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
